Entraînement ResNet50


1. Entraînement ResNet50 avec OCT 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from tqdm import tqdm

In [3]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# ── Une seule transformation ───────────────────────────────
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ══ OCT ═══════════════════════════════════════════════════

oct_train_loader = DataLoader(
    datasets.ImageFolder("../data/OCT_aug/train", transform=transform),
    batch_size=BATCH_SIZE, shuffle=True, 
)
oct_val_loader = DataLoader(
    datasets.ImageFolder("../data/OCT_aug/val",   transform=transform),
    batch_size=BATCH_SIZE, shuffle=False
)
oct_test_loader = DataLoader(
    datasets.ImageFolder("../data/OCT_aug/test",  transform=transform),
    batch_size=BATCH_SIZE, shuffle=False
)

# ══ X-RAY ═════════════════════════════════════════════════

xray_train_loader = DataLoader(
    datasets.ImageFolder("../data/xray_aug/train", transform=transform),
    batch_size=BATCH_SIZE, shuffle=True
)
xray_val_loader = DataLoader(
    datasets.ImageFolder("../data/xray_aug/val",   transform=transform),
    batch_size=BATCH_SIZE, shuffle=False
)
xray_test_loader = DataLoader(
    datasets.ImageFolder("../data/xray_aug/test",  transform=transform),
    batch_size=BATCH_SIZE, shuffle=False
)

print("✅ DataLoaders créés !")
print(f"OCT   classes : {oct_train_loader.dataset.class_to_idx}")
print(f"X-Ray classes : {xray_train_loader.dataset.class_to_idx}")
print(f"OCT   Train   : {len(oct_train_loader.dataset)} images")
print(f"OCT   Val     : {len(oct_val_loader.dataset)} images")
print(f"OCT   Test    : {len(oct_test_loader.dataset)} images")
print(f"X-Ray Train   : {len(xray_train_loader.dataset)} images")
print(f"X-Ray Val     : {len(xray_val_loader.dataset)} images")
print(f"X-Ray Test    : {len(xray_test_loader.dataset)} images")

✅ DataLoaders créés !
OCT   classes : {'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3}
X-Ray classes : {'NORMAL': 0, 'PNEUMONIA': 1}
OCT   Train   : 128393 images
OCT   Val     : 9229 images
OCT   Test    : 15373 images
X-Ray Train   : 8658 images
X-Ray Val     : 699 images
X-Ray Test    : 1165 images


1. Construire ResNet50

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")


Device utilisé : cpu


In [8]:
def build_resnet50(num_classes, device):
    """
    Transfer Learning ResNet50 :
    1. Charger poids pré-entraînés ImageNet
    2. Geler toutes les couches
    3. Remplacer la dernière couche
    """

    # ── Charger ResNet50 pré-entraîné ─────────────────────
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    # ── Geler toutes les couches ───────────────────────────
    for param in model.parameters():
        param.requires_grad = False

    # ── Remplacer la dernière couche ───────────────────────
    in_features = model.fc.in_features  # 2048
    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes)
    )

    model = model.to(device)

    # ── Résumé ─────────────────────────────────────────────
    total_params    = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters()
                          if p.requires_grad)

    print(f"✅ ResNet50 chargé !")
    print(f"   Total params     : {total_params:,}")
    print(f"   Trainable params : {trainable_params:,}")
    print(f"   Frozen params    : {total_params - trainable_params:,}")
    print(f"   Dernière couche  : {model.fc}")

    return model

# ── OCT — 4 classes ───────────────────────────────────────
resnet_oct = build_resnet50(num_classes=4, device=device)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\DELL/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [03:41<00:00, 463kB/s] 


✅ ResNet50 chargé !
   Total params     : 24,559,172
   Trainable params : 1,051,140
   Frozen params    : 23,508,032
   Dernière couche  : Sequential(
  (0): Linear(in_features=2048, out_features=512, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=512, out_features=4, bias=True)
)


Définir Loss et optimiser 

In [10]:
# ── Loss ──────────────────────────────────────────────────
criterion_oct = nn.CrossEntropyLoss()

# ── Optimizer — seulement les couches non gelées ──────────
optimizer_oct = optim.Adam(
    filter(lambda p: p.requires_grad, resnet_oct.parameters()),
    lr=1e-3
)

# ── Scheduler — réduire lr si val_loss stagne ─────────────
scheduler_oct = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_oct,
    mode    = 'min',
    factor  = 0.5,
    patience= 3,
)

print("✅ Loss, Optimizer, Scheduler définis !")
print(f"   Loss      : CrossEntropyLoss")
print(f"   Optimizer : Adam (lr=1e-3)")
print(f"   Scheduler : ReduceLROnPlateau (patience=3)")

✅ Loss, Optimizer, Scheduler définis !
   Loss      : CrossEntropyLoss
   Optimizer : Adam (lr=1e-3)
   Scheduler : ReduceLROnPlateau (patience=3)


3.  Fonction d'entraînement

In [11]:
def train_epoch(model, loader, criterion, optimizer, device):
    """
    Une epoch d'entraînement
    Retourne : loss moyenne, accuracy
    """
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        # Forward
        outputs = model(images)
        loss    = criterion(outputs, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Métriques
        total_loss += loss.item()
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

4.Fonction d'évaluation

In [12]:
def eval_epoch(model, loader, criterion, device):
    """
    Évaluation sur val ou test
    Retourne : loss moyenne, accuracy
    """
    model.eval()
    total_loss = 0
    correct    = 0
    total      = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Eval", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss    = criterion(outputs, labels)

            total_loss += loss.item()
            preds       = outputs.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

5. Boucle d'entraînement complète


In [13]:
def entrainer_modele(model, train_loader, val_loader,
                     criterion, optimizer, scheduler,
                     num_epochs, device, save_path):
    """
    Entraînement complet avec :
    - Early stopping
    - Sauvegarde du meilleur modèle
    - Historique des métriques
    """

    historique = {
        'train_loss': [], 'train_acc': [],
        'val_loss'  : [], 'val_acc'  : []
    }

    meilleure_val_loss = float('inf')
    patience_counter   = 0
    EARLY_STOP_PATIENCE = 7

    print(f"{'='*60}")
    print(f"  Début entraînement — {num_epochs} epochs max")
    print(f"  Save path : {save_path}")
    print(f"{'='*60}\n")

    for epoch in range(num_epochs):

        # ── Train ──────────────────────────────────────────
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # ── Validation ─────────────────────────────────────
        val_loss, val_acc = eval_epoch(
            model, val_loader, criterion, device
        )

        # ── Scheduler ──────────────────────────────────────
        scheduler.step(val_loss)

        # ── Sauvegarder historique ──────────────────────────
        historique['train_loss'].append(train_loss)
        historique['train_acc'].append(train_acc)
        historique['val_loss'].append(val_loss)
        historique['val_acc'].append(val_acc)

        # ── Affichage ───────────────────────────────────────
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f} "
              f"Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} "
              f"Val Acc: {val_acc:.4f}")

        # ── Sauvegarder meilleur modèle ─────────────────────
        if val_loss < meilleure_val_loss:
            meilleure_val_loss = val_loss
            patience_counter   = 0
            torch.save(model.state_dict(), save_path)
            print(f"  ✅ Meilleur modèle sauvegardé ! "
                  f"(val_loss={val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"  ⚠️  Pas d'amélioration "
                  f"({patience_counter}/{EARLY_STOP_PATIENCE})")

        # ── Early Stopping ──────────────────────────────────
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n🛑 Early stopping à l'epoch {epoch+1}")
            break

    print(f"\n{'='*60}")
    print(f"  Entraînement terminé !")
    print(f"  Meilleure val_loss : {meilleure_val_loss:.4f}")
    print(f"{'='*60}")

    return historique

6 .Lancer l'entraînement Phase 1


In [ ]:
os.makedirs("../models", exist_ok=True)

# ── Phase 1 : Feature Extraction (couches gelées) ──────────
print("PHASE 1 — Feature Extraction")
print("Toutes les couches ResNet gelées")
print("Seul le classifier final est entraîné\n")

historique_oct_resnet = entrainer_modele(
    model        = resnet_oct,
    train_loader = oct_train_loader,
    val_loader   = oct_val_loader,
    criterion    = criterion_oct,
    optimizer    = optimizer_oct,
    scheduler    = scheduler_oct,
    num_epochs   = 10,
    device       = device,
    save_path    = "../models/resnet50_oct_phase1.pth"
)

PHASE 1 — Feature Extraction
Toutes les couches ResNet gelées
Seul le classifier final est entraîné

  Début entraînement — 10 epochs max
  Save path : ../models/resnet50_oct_phase1.pth



Train:  27%|██▋       | 1095/4013 [1:15:47<3:17:31,  4.06s/it]

7. Phase 2 : Fine-Tuning

In [ ]:
# ── Dégeler les 30 dernières couches ──────────────────────
print("PHASE 2 — Fine-Tuning")
print("Dégel des 30 dernières couches\n")

# Dégeler les dernières couches
layers = list(resnet_oct.children())
for layer in layers[-3:]:
    for param in layer.parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in resnet_oct.parameters()
                if p.requires_grad)
print(f"Params entraînables après dégel : {trainable:,}")

# Nouvel optimizer avec lr plus petit
optimizer_oct_ft = optim.Adam(
    filter(lambda p: p.requires_grad, resnet_oct.parameters()),
    lr=1e-5     # lr plus petit pour fine-tuning
)

scheduler_oct_ft = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_oct_ft,
    mode='min', factor=0.5, patience=3, verbose=True
)

historique_oct_resnet_ft = entrainer_modele(
    model        = resnet_oct,
    train_loader = oct_train_loader,
    val_loader   = oct_val_loader,
    criterion    = criterion_oct,
    optimizer    = optimizer_oct_ft,
    scheduler    = scheduler_oct_ft,
    num_epochs   = 20,
    device       = device,
    save_path    = "../models/resnet50_oct_best.pth"
)